# NB01 Data Collection
## Sector Rotation & Federal Reserve Rate Cycles
### DS105W Data for Data Science | Group Project 2025–2026

<div style="font-family:system-ui; padding:16px 24px; background:#FFFFFF;
     border-left:8px solid #ED9255; border-radius:8px;
     box-shadow:0 4px 12px rgba(0,0,0,0.08); max-width:640px; color:#212121">

**Team:** Git It Done

| Member | Role | Notebook |
|--------|------|----------|
| **Hugo Whyte** | Data Collection | **This notebook** |
| Parthiv | Cleaning & Database | NB02 |
| Joel Saldanha | Analysis & Website | NB03 + GitHub Pages |

**Research Question:** *How do US equity sector returns vary across Federal Reserve rate hiking and cutting cycles, and which sectors consistently rotate into outperformance as monetary policy shifts?*

</div>

## Decisions Made Before Writing Any Code

Following the workflow taught in W03, I document all data-collection decisions *before* writing
any code so the choices are transparent and reproducible.

---

### 1 · Data Sources

| Data | Source | Series / Endpoint |
|------|--------|-------------------|
| Federal Funds Rate | FRED (St. Louis Fed) | `FEDFUNDS` |
| CPI Inflation | FRED | `CPIAUCSL` |
| 10Y–2Y Yield Spread | FRED | `T10Y2Y` |
| S&P 500 Sector ETF prices | Alpha Vantage | `TIME_SERIES_MONTHLY_ADJUSTED` |

Both APIs are free, require only an API key, and return JSON, the format taught in W02/W03.
All requests use Python's `requests` library directly, with no pre-built wrappers.

---

### 2 · Time Horizon

**FRED macro series:** January 1993 – present (`observation_start=1993-01-01`).
This captures five full hiking cycles (1994–95, 1999–2000, 2004–06, 2015–18, 2022–23)
and multiple cutting cycles, giving robust statistical coverage.

**ETF prices:** The original nine SPDR sector ETFs launched in December 1998, so usable
monthly price history starts from **January 2000**. XLRE (Real Estate) launched in
**October 2015**, it is collected from that date only; the shorter history is noted
explicitly and handled transparently in NB02.

---

### 3 · ETF Universe (10 sectors)

| Ticker | Sector | Available from |
|--------|--------|----------------|
| XLK | Technology | ~Jan 2000 |
| XLF | Financials | ~Jan 2000 |
| XLE | Energy | ~Jan 2000 |
| XLV | Health Care | ~Jan 2000 |
| XLU | Utilities | ~Jan 2000 |
| XLY | Consumer Discretionary | ~Jan 2000 |
| XLP | Consumer Staples | ~Jan 2000 |
| XLI | Industrials | ~Jan 2000 |
| XLB | Materials | ~Jan 2000 |
| XLRE | Real Estate | Oct 2015 only |

XLRE is included because Real Estate is among the sectors most sensitive to interest rates,
making it analytically valuable despite its shorter history.

---

### 4 · API Key Security

Both keys are stored in a `.env` file at the repository root and loaded with `python-dotenv`
and `os.getenv()`, the same pattern used in Mini-Project 1 (W03). The `.env` file is listed
in `.gitignore` so keys are never pushed to GitHub.

---

### 5 · Rate Limiting

Alpha Vantage free tier: **25 requests/day**, ~5/minute.
With 10 ETFs + 3 FRED calls = 13 total, we are safely within the daily limit.
A `time.sleep(15)` pause is inserted between every Alpha Vantage call to avoid burst limits.
Total collection time: ~2.5 minutes. This pattern was taught in W03.

---

### 6 · Save-Before-Transform Rule

Every raw JSON response is saved to `data/raw/` **immediately** after collection, before any
transformation. This means the APIs only need to be called once, NB02 and NB03 read from
disk. Taught explicitly in W03.

In [7]:
# Step 1: Load Libraries
import os
import json
import time
import requests
from dotenv import load_dotenv

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [8]:
# Step 2: Load API Keys

# load_dotenv() reads the .env file and makes its contents available via os.getenv()
load_dotenv()

# Retrieve both keys, we NEVER print the actual values
FRED_KEY = os.getenv("FRED_API_KEY")
AV_KEY   = os.getenv("ALPHAVANTAGE_API_KEY")

# Guard: stop immediately if either key is missing
if not FRED_KEY:
    raise ValueError("FRED_API_KEY not found. Check your .env file.")
if not AV_KEY:
    raise ValueError("ALPHAVANTAGE_API_KEY not found. Check your .env file.")

print("FRED API key      : loaded successfully!")
print("Alpha Vantage key : loaded successfully!")

FRED API key      : loaded successfully!
Alpha Vantage key : loaded successfully!


## Step 3: Create Output Folder

Following the repository structure in the README, all raw JSON files go to `data/raw/`.

`os.makedirs(..., exist_ok=True)` creates the folder if it doesn't exist, and does nothing
if it already does, so this cell is safe to re-run at any time (W03).

In [15]:
# Notebooks run with their working directory set to the notebooks/ folder.
# We navigate one level up (.. ) to reach the repo root, then build the path
# to data/raw/ from there. This keeps all raw data at the repo root as per the README.
# os.path.abspath() converts the relative .. into a full absolute path (W03).

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DIR   = os.path.join(REPO_ROOT, "data", "raw")

# exist_ok=True: create if missing, silently do nothing if already there
os.makedirs(RAW_DIR, exist_ok=True)

print(f"Repo root  : {REPO_ROOT}")
print(f"Raw data   : {RAW_DIR}")

Repo root  : /files/group-project-git-it-done
Raw data   : /files/group-project-git-it-done/data/raw


## Step 4: Collect FRED Macro Data

The FRED API returns JSON and requires a `series_id` and `api_key` parameter.
The full URL looks like:https://api.stlouisfed.org/fred/series/observations?series_id=FEDFUNDS&api_key=YOUR_KEY&file_type=json&observation_start=1993-01-01

We collect three series using a **loop over a dictionary** (W02). The dictionary maps
each `series_id` to its output filename, this keeps the code clean and easy to extend.

Each response is saved to disk immediately after the request (W03 save-before-transform rule).

In [10]:
# FRED base URL
FRED_BASE = "https://api.stlouisfed.org/fred/series/observations"

# Dictionary: series_id → output filename (dictionaries store key-value pairs)
FRED_SERIES = {
    "FEDFUNDS": "fred_fedfunds.json",   # Federal Funds Rate
    "CPIAUCSL": "fred_cpiaucsl.json",   # CPI Inflation
    "T10Y2Y":   "fred_t10y2y.json",     # 10Y-2Y Yield Spread
}

for series_id, filename in FRED_SERIES.items():

    print(f"Requesting {series_id}...", end=" ")

    # Parameters dictionary, requests.get() builds the full URL from this (W02)
    params = {
        "series_id":         series_id,
        "api_key":           FRED_KEY,
        "file_type":         "json",
        "observation_start": "1993-01-01",
        "frequency":         "m",          # Monthly
    }

    response = requests.get(FRED_BASE, params=params)

    # 200 = success; anything else = problem (W02)
    if response.status_code != 200:
        print(f"FAILED — status {response.status_code}")
        print(response.text)
        continue

    data = response.json()

    # Save raw JSON immediately, before any transformation (W03)
    output_path = os.path.join(RAW_DIR, filename)
    with open(output_path, "w") as f:
        json.dump(data, f, indent=2)

    n = len(data["observations"])
    print(f"OK — {n} observations → {output_path}")

    time.sleep(1)   # Polite pause between FRED calls

print("\nAll FRED series collected.")

Requesting FEDFUNDS... OK — 399 observations → data/raw/fred_fedfunds.json
Requesting CPIAUCSL... OK — 399 observations → data/raw/fred_cpiaucsl.json
Requesting T10Y2Y... OK — 400 observations → data/raw/fred_t10y2y.json

All FRED series collected.


## Step 5: Inspect FRED Response Structure

Before moving on, we reload the FEDFUNDS file from disk and peek at its structure.
This is the "inspect before transform" habit from W03, understanding the JSON shape
now means Parthiv can write NB02 transformations without guessing.

We reload from disk (not from the live response variable) to also confirm the save worked.

In [11]:
# Reload from disk, this also confirms the save worked correctly (W03)
with open(os.path.join(RAW_DIR, "fred_fedfunds.json"), "r") as f:
    fedfunds_check = json.load(f)

print("Top-level keys:", list(fedfunds_check.keys()))

observations = fedfunds_check["observations"]
print(f"Number of monthly observations: {len(observations)}")

# Preview first and last record to understand the structure
print("\nFirst record:", observations[0])
print("Last record :", observations[-1])

# Note for NB02: 'value' is a string, will need converting to float
print("\nNote: 'value' field is a string,  Parthiv will convert to float in NB02.")

Top-level keys: ['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations']
Number of monthly observations: 399

First record: {'realtime_start': '2026-04-20', 'realtime_end': '2026-04-20', 'date': '1993-01-01', 'value': '3.02'}
Last record : {'realtime_start': '2026-04-20', 'realtime_end': '2026-04-20', 'date': '2026-03-01', 'value': '3.64'}

Note: 'value' field is a string,  Parthiv will convert to float in NB02.


## Step 6: Collect Alpha Vantage ETF Price Data

Alpha Vantage's `TIME_SERIES_MONTHLY_ADJUSTED` endpoint returns adjusted monthly closing
prices. We use **adjusted** prices (not raw) because they account for dividends and splits,
giving a true picture of total return, this is the standard in finance.

The endpoint URL looks like:https://www.alphavantage.co/query?function=TIME_SERIES_MONTHLY_ADJUSTED&symbol=XLK&apikey=YOUR_KEY

**Rate limiting:** `time.sleep(15)` between each call, ~2.5 minutes total for 10 ETFs.
The cell prints progress as it goes so you can see it working.

**XLRE note:** XLRE launched October 2015, Alpha Vantage returns all available history
automatically, so XLRE will simply have fewer records than the others. This is expected.

In [ ]:
AV_BASE = "https://www.alphavantage.co/query"

# Dictionary: ticker → (output filename, description note)
ETFS = {
    "XLK":  ("av_XLK.json",  "Technology"),
    "XLF":  ("av_XLF.json",  "Financials"),
    "XLE":  ("av_XLE.json",  "Energy"),
    "XLV":  ("av_XLV.json",  "Health Care"),
    "XLU":  ("av_XLU.json",  "Utilities"),
    "XLY":  ("av_XLY.json",  "Consumer Discretionary"),
    "XLP":  ("av_XLP.json",  "Consumer Staples"),
    "XLI":  ("av_XLI.json",  "Industrials"),
    "XLB":  ("av_XLB.json",  "Materials"),
    "XLRE": ("av_XLRE.json", "Real Estate — from Oct 2015 only"),
}

for i, (ticker, (filename, note)) in enumerate(ETFS.items()):

    print(f"[{i+1}/{len(ETFS)}] {ticker} ({note})...", end=" ")

    params = {
        "function": "TIME_SERIES_MONTHLY_ADJUSTED",
        "symbol":   ticker,
        "apikey":   AV_KEY,
    }

    response = requests.get(AV_BASE, params=params)

    if response.status_code != 200:
        print(f"FAILED — HTTP {response.status_code}")
        continue

    data = response.json()

    # Alpha Vantage signals a rate limit with an 'Information' key
    if "Information" in data:
        print("RATE LIMIT HIT: wait 60 seconds, then re-run from this ticker.")
        print(data["Information"])
        break

    if "Error Message" in data:
        print(f"ERROR — {data['Error Message']}")
        continue

    # Save raw JSON immediately (W03)
    output_path = os.path.join(RAW_DIR, filename)
    with open(output_path, "w") as f:
        json.dump(data, f, indent=2)

    ts = data.get("Monthly Adjusted Time Series", {})
    dates = sorted(ts.keys())
    print(f"OK — {len(ts)} months  ({dates[0]} → {dates[-1]})")

    # 15-second pause between requests — skip after the last one
    if i < len(ETFS) - 1:
        print(f"   (waiting 15s...)")
        time.sleep(15)

print("\nAll ETF data collection complete.")

[1/10] XLK (Technology)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[2/10] XLF (Financials)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[3/10] XLE (Energy)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[4/10] XLV (Health Care)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[5/10] XLU (Utilities)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[6/10] XLY (Consumer Discretionary)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[7/10] XLP (Consumer Staples)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[8/10] XLI (Industrials)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[9/10] XLB (Materials)... OK — 317 months  (1999-12-31 → 2026-04-17)
   (waiting 15s...)
[10/10] XLRE (Real Estate — from Oct 2015 only)... OK — 126 months  (2015-11-30 → 2026-04-17)

All ETF data collection complete.


## Step 7: Inspect One ETF Response Structure

We reload XLK from disk and preview the structure.
This tells Parthiv exactly what keys to expect in NB02 when flattening the JSON.

In [13]:
with open(os.path.join(RAW_DIR, "av_XLK.json"), "r") as f:
    xlk_check = json.load(f)

print("Top-level keys:", list(xlk_check.keys()))

ts = xlk_check["Monthly Adjusted Time Series"]
dates = sorted(ts.keys())

print(f"\nMonthly records : {len(ts)}")
print(f"Earliest month  : {dates[0]}")
print(f"Most recent     : {dates[-1]}")

# Show all fields in one record so NB02 knows exactly what to expect
print(f"\nAll fields in latest record ({dates[-1]}):")
for field, value in ts[dates[-1]].items():
    print(f"  '{field}': {value}")

print("\nKey field for return calculations: '5. adjusted close'")

Top-level keys: ['Meta Data', 'Monthly Adjusted Time Series']

Monthly records : 317
Earliest month  : 1999-12-31
Most recent     : 2026-04-17

All fields in latest record (2026-04-17):
  '1. open': 134.1150
  '2. high': 154.8050
  '3. low': 131.3546
  '4. close': 154.3500
  '5. adjusted close': 154.3500
  '6. volume': 132575615
  '7. dividend amount': 0.0000

Key field for return calculations: '5. adjusted close'


## Step 8: Full Verification

We verify every single file: does it exist, does it have data, and does the date range
look correct? This "reload-and-check" step was taught in W03 and used in Mini-Project 1.
If anything looks wrong here, we fix it before handing off to NB02.

In [14]:
print("=" * 68)
print("VERIFICATION - All Raw Files")
print("=" * 68)

# ── FRED ────────────────────────────────────────────────────────────────────
print("\n[ FRED Macro Series ]")
fred_files = {
    "fred_fedfunds.json": "FEDFUNDS",
    "fred_cpiaucsl.json": "CPIAUCSL",
    "fred_t10y2y.json":   "T10Y2Y",
}
for filename, series in fred_files.items():
    with open(os.path.join(RAW_DIR, filename), "r") as f:
        d = json.load(f)
    obs = d["observations"]
    print(f"  {series:<12}  {len(obs):>4} records   {obs[0]['date']} → {obs[-1]['date']}")

# ── Alpha Vantage ────────────────────────────────────────────────────────────
print("\n[ Alpha Vantage Sector ETFs ]")
for ticker, (filename, note) in ETFS.items():
    with open(os.path.join(RAW_DIR, filename), "r") as f:
        d = json.load(f)
    ts    = d["Monthly Adjusted Time Series"]
    dates = sorted(ts.keys())
    print(f"  {ticker:<6}  {len(ts):>4} months   {dates[0]} → {dates[-1]}   {note}")

print("\n" + "=" * 68)
print("All files verified. Ready to hand off to Parthiv (NB02).")
print("=" * 68)

VERIFICATION - All Raw Files

[ FRED Macro Series ]
  FEDFUNDS       399 records   1993-01-01 → 2026-03-01
  CPIAUCSL       399 records   1993-01-01 → 2026-03-01
  T10Y2Y         400 records   1993-01-01 → 2026-04-01

[ Alpha Vantage Sector ETFs ]
  XLK      317 months   1999-12-31 → 2026-04-17   Technology
  XLF      317 months   1999-12-31 → 2026-04-17   Financials
  XLE      317 months   1999-12-31 → 2026-04-17   Energy
  XLV      317 months   1999-12-31 → 2026-04-17   Health Care
  XLU      317 months   1999-12-31 → 2026-04-17   Utilities
  XLY      317 months   1999-12-31 → 2026-04-17   Consumer Discretionary
  XLP      317 months   1999-12-31 → 2026-04-17   Consumer Staples
  XLI      317 months   1999-12-31 → 2026-04-17   Industrials
  XLB      317 months   1999-12-31 → 2026-04-17   Materials
  XLRE     126 months   2015-11-30 → 2026-04-17   Real Estate — from Oct 2015 only

All files verified. Ready to hand off to Parthiv (NB02).


## Summary

In this notebook I:

1. **Documented all decisions** before writing any code: sources, time horizons, ETF universe, key security, and rate limiting
2. **Loaded API keys securely** from a `.env` file: keys never hardcoded, never pushed to GitHub
3. **Collected 3 FRED macro series** via a loop over a dictionary using `requests.get()`
4. **Collected 10 Alpha Vantage ETF price series** with `time.sleep(15)` between calls
5. **Saved all 13 raw JSON files** to `data/raw/` immediately after collection
6. **Inspected the raw structure** of both API response types to inform NB02
7. **Verified all files** by reloading from disk and checking record counts and date ranges

---

**Next step → NB02 (Parthiv):** Flatten JSON with `pd.json_normalize()`, label rate cycle
periods using a custom function + `.apply()`, merge macro + sector data on `date`,
and write to SQLite.